# Devoir maison 1 — carnets J1 et J2

> **À rendre avant le 19 octobre** — et, en général, **un devoir se rend avant la première séance de la
> semaine suivante**. Ce devoir porte sur les carnets **J1** (du fichier brut à la
> soumission) et **J2** (ce que la table cache). Il se fait **seul**, à la maison, et il se rend.

**Ce qu'on rend :** ce carnet, rempli, déposé dans votre **dépôt GitHub personnel** avec la
déclaration d'usage de l'IA. Les trois tableaux de la fin sont la pièce principale.

**Pourquoi ce devoir :** chaque question demande une mesure que **vous** produisez — pas une
lecture. Si vous n'avez pas compris, ça se verra tout de suite ; c'est fait pour ça.

**Trois mots, pour être sûr :**

- **les deux réponses** — le patient vient (`patient_absent` vaut 0), ou il ne vient pas (1) ;
- **la référence** — le score d'un modèle qui répond toujours la même chose : le minimum à battre ;
- **un constat** — une observation, un chiffre, une comparaison, et ce que vous en faites.

**Treize questions, dont dix de cœur.** Chaque titre porte sa mention : **cœur** = le socle du
semestre, à faire dans tous les cas ; **approfondissement** = à faire dès que le cœur tient, et
à reprendre plus tard si le temps manque — mais rien n'est facultatif pour autant : les trois
approfondissements portent les compétences les plus fines (contrôler un fichier, casser une
interface, écrire une feature, expliquer une valeur, démêler une confusion).

---

## Comment ça marche

1. La partie **« Mise en place »** est **fournie** : exécutez-la de haut en bas. **Une seule de ses
   cellules est faite pour être modifiée** — `ajouter_features`, la question 9 le demande.
2. Chaque **question** porte sa consigne, en markdown, et un **emplacement vide** à remplir.
3. Les questions sont dans l'ordre des carnets : **1 à 7 sur J1**, **8 à 13 sur J2**.
4. Une réponse se **chiffre** ou s'**écrit** — jamais « ça a marché ». Plusieurs questions
   demandent d'**annoncer ce que vous attendez AVANT de mesurer** : `noter(...)` refuse
   d'ailleurs d'enregistrer un essai non annoncé.
5. À la fin : les **trois tableaux**, puis le **dépôt**.

**Ce qui est lu.** Ce qui est lu ici, ce sont **les trois tableaux** — constats, hypothèses,
essais — et le code, là où il produit un chiffre. **Ce dépôt est relevé** : il est lu, repris à
l'oral, et **les sept devoirs comptent ensemble pour 10 % de la note** — dans la composante
« exercices et devoirs » du cours (25 %, avec E1, E2, E3). Et l'oral revient sur votre tableau d'essais :
**venez en séance 3 avec lui**, deux cellules seront inspectées à voix haute.

> **La règle, la même qu'en séance :** un score sans référence ne veut rien dire, et un constat
> sans chiffre n'en est pas un.


## Mise en place (fournie)

Exécutez cette partie **de haut en bas**, sans rien y modifier. Elle est **extraite des carnets
J1 et J2** : les données, la table de travail, la référence, le modèle — et les objets dont les
questions ont besoin.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5) :
le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [ ]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [ ]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"

print("le dossier des données trouvé :", DOSSIER)


In [ ]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"

print("les deux tableaux \n")
print("train :", train.shape, "(lignes, colonnes)")
print("test  :", test.shape, "(lignes, colonnes)")

print("les cinq premières lignes \n")
display(train.head())


### La table de travail


In [ ]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)

print("la table de travail \n")
print("colonnes conservées :", X.shape[1])
print("colonnes écartées (cardinalité trop forte) :", TROP_FIN)
print("colonnes catégorielles :", CAT)


### La cible, et la référence


In [ ]:
print("la distribution de la cible \n")
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])
display(repartition)

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())
print(f"la référence (répondre toujours la classe majoritaire) : {reference:.2f} %")


In [ ]:
y = train[CIBLE]
reference = 100 * max(y.mean(), 1 - y.mean())

print(f"référence (classe majoritaire) : {reference:.2f} % d'exactitude")


### Le premier modèle


In [ ]:
# le modèle et le découpage : ils entrent ici
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
score_interne = 100 * modele.score(Xval, yval)

print(f"exactitude de validation : {score_interne:.2f} %   (référence : {reference:.2f} %)")
print(f"arbres effectivement construits : {modele.n_iter_} — le modèle s'arrête TOUT SEUL dès que sa")
print("  part interne cesse de progresser (c'est l'arrêt anticipé), et non au nombre demandé.")


### Vos features — LA cellule que vous modifiez


In [ ]:
# votre feature : elle reçoit la TABLE DE TRAVAIL — identifiants, dates et colonnes post-issue
# déjà retirés — et la rend enrichie. `predict` l'appelle sur la table BRUTE, après avoir retiré
# les mêmes colonnes : le crochet voit donc la même forme des deux côtés, ce qui est la règle.
def ajouter_features(df):
    """Vos features maison. Appliquée identiquement à l'entraînement et au test."""
    df = df.copy()
    # Exemple — remplacez-le par la vôtre :
    # df["plus_de_60_ans"] = (df["age"] > 60).astype(int)
    return df

# Votre feature entre par `preparer()`, la chaîne unique : on n'applique RIEN à la main.
# Après l'avoir écrite, appelez `remesurer()` (au devoir, dans la cellule prévue) : `X` la
# reçoit une seule fois, et `predict` l'applique aussi, de son côté, sur la table brute.


### La chaîne — une seule, des deux côtés


In [ ]:
# ── LA CHAÎNE : une seule fonction, celle que `predict` ET l'entraînement utilisent.
def preparer(brut):
    """Écarte, type, applique vos features : LA chaîne, des deux côtés.
       `predict` l'appelle ; l'entraînement l'appelle aussi, par `matrice()` sur la table de
       travail. Contourner cette fonction, c'est apprendre sur une table et prédire sur une
       autre — exactement le défaut que la séance 2 traque."""
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns],
                  errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis X : ré-exécutable sans que la
       feature se superpose à elle-même."""
    global modele
    Xc = ajouter_features(X)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)


### L'interface — le modèle qu'on peut appeler


In [ ]:
# numpy, pour la seule assertion qui suit
import numpy as np

def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours appellera.
       Elle passe par `preparer(...)`, la même chaîne que l'entraînement — pas une copie."""
    # elle ne refait PAS la chaîne à la main : elle appelle `preparer`, celle de l'entraînement.
    # C'est ce couplage qui garantit qu'une table n'est jamais préparée de deux façons.
    return modele.predict(preparer(df)).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"
    print(f"test OK — 500 prédictions, {int(sum(p))} positifs")

test_predict()


### Le tableau d'essais


In [ ]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

# les deux amorces du carnet sont annoncées elles aussi — sinon elles ne passeraient pas
prevoir("rien faire du tout", "c'est le minimum à battre",
        "un modèle qui répond toujours la même chose vaut la classe majoritaire")
prevoir("un modèle suffit-il ?", "mieux que la référence",
        "les arbres captent une partie du signal, pas tout")
noter("rien faire du tout", "classe majoritaire", "c'est le minimum à battre", reference)
noter("un modèle suffit-il ?", "HistGradientBoosting", "mieux que la référence", score_interne)


### La soumission


In [ ]:
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)

print("le fichier écrit : submission.csv \n")
print("dimensions :", soumission.shape, "(lignes, colonnes)")
display(soumission.head())


### Le modèle linéaire


In [ ]:
# le modèle linéaire : il entre ici
from sklearn.linear_model import LogisticRegression
X_num = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xval_n = X_num.loc[Xtr.index], X_num.loc[Xval.index]

modele_lineaire = LogisticRegression(solver="newton-cholesky").fit(Xtr_n, ytr)
score_lineaire = 100 * modele_lineaire.score(Xval_n, yval)


### Mesurer une colonne


In [ ]:
def mesurer(avec_colonne):
    """La même mesure, avec ou sans la colonne que le test ne portera plus."""
    Xa = X.copy()
    if avec_colonne:
        Xa["canal_prise"] = train.loc[Xa.index, "canal_prise"].fillna("").astype("category")
    a, b, c, d = train_test_split(Xa, y, test_size=0.2, stratify=y, random_state=0)
    return 100 * HistGradientBoostingClassifier(random_state=0).fit(a, c).score(b, d)


### Le premier graphique


In [ ]:
# matplotlib, pour les graphiques des questions 2 et 5
import matplotlib.pyplot as plt


### Après avoir écrit votre feature (question 9)


In [ ]:
# votre feature entre par la chaîne unique : on n'applique rien à la main.
# relancez CETTE cellule quand vous l'avez écrite — `X` la reçoit une seule fois,
# `modele` est mis à jour, et `predict` l'applique aussi de son côté.
print(f"exactitude de validation, table de travail courante : {remesurer():.2f} %")


### Les gabarits des constats, des hypothèses et des essais

**Un constat qui ne dit rien, et le même qui dit tout.** C'est le contraste qui apprend, plus
que trois bons exemples d'affilée :

| | |
|---|---|
| ✗ **à jeter** | « il y a des valeurs manquantes » |
| | pas de chiffre, pas de comparaison, aucune conséquence — on ne peut rien en faire, et
  personne ne peut le contredire |
| ✓ **à garder** | « `motif_absence` est vide pour **100 %** des présents et renseignée pour
  **100 %** des absents, contre 5,5 % de trous sur `solde_credit` : la colonne dit la réponse,
  donc elle sort » |
| | un chiffre, une comparaison, une conséquence — et un lecteur qui sait quoi en faire |

**Les deux questions qui séparent les deux :** *par rapport à quoi ?* et *qu'est-ce que ça
change ?* Un constat qui ne répond pas aux deux est une impression.

**Et vous devez pouvoir le défendre à l'oral** : chaque cellule que vous rendez, vous devez
savoir l'expliquer sans la relire. Le test, à faire à deux : expliquez-la à un camarade qui ne
l'a pas vue. S'il repose une question, elle n'est pas encore claire — pour lui comme pour vous.


In [ ]:
# les trois tableaux, et les outils qui les remplissent

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? · conséquence : garder, écarter, surveiller.
       Ré-exécuter une cellule REMPLACE la ligne de même `observation`."""
    ligne = {"observation": observation, "chiffre": chiffre,
             "comparaison": comparaison, "conséquence": consequence}
    CONSTATS[:] = [c for c in CONSTATS if c["observation"] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester.
       Ré-exécuter une cellule REMPLACE la ligne de même `feature`."""
    ligne = {"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
             "test": comment_on_la_test, "contredite si": ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h["feature"] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

# la barre de bruit, calculée SUR LE JEU DE VALIDATION DE 20 % — c'est écrit ici parce que la
# question 5 en change la taille, et la barre change avec elle.
erreur_type = 100 * ((score_interne / 100) * (1 - score_interne / 100) / len(yval)) ** 0.5
print(f"erreur-type de l'exactitude de validation (jeu de 20 %) : ± {erreur_type:.2f} point(s)")
print("  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.")

VERIFICATIONS = []

def verifier(nom, condition, indice=''):
    """Contrôle silencieux : il ne dit JAMAIS la valeur attendue, seulement si la condition
       tient. Recopier ce message ne donne ni la valeur ni la réponse attendue."""
    VERIFICATIONS.append({'contrôle': nom, 'tient': bool(condition)})
    if condition:
        print('  ok        ' + nom)
    else:
        print('  À REVOIR  ' + nom + '   ->   ' + indice)

def controle(nom, quoi, indice=''):
    """Contrôle de FORME, posé dans la cellule de la question. Il ne compare JAMAIS à une valeur
       attendue : il vérifie que la réponse a la forme qu'elle doit avoir — un type, une plage,
       une dimension, ou une cohérence avec VOS autres réponses. Tant que la réponse n'est pas
       écrite, il se tait.

       `quoi` est une fonction sans argument qui rend True ou False (une lambda)."""
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))


## Les questions

Seize questions. Les dix premières portent sur **J1**, les six dernières sur **J2**. Pour
chacune : lisez la consigne, écrivez votre code (ou votre phrase) à l'emplacement prévu, et
**exécutez la cellule**.

La consigne indique toujours **ce qu'on écrit** : un chiffre, une phrase, ou une ligne de
tableau. C'est cela qui est lu.


### Question 1 — la cible, et la référence  ·  *carnet J1*  ·  *cœur*


La référence, recalculée et **prouvée**. C'est le chiffre qui décide si tout ce qui suit vaut
quelque chose.

À faire :
  1. recalculez la référence en UNE ligne, sans réutiliser la variable `reference` de la mise en
     place, et comparez votre résultat à celui que la séance 1 obtient ;
  2. construisez un modèle qui répond TOUJOURS la même chose (un tableau rempli de la valeur la
     plus fréquente) et montrez qu'il **atteint** cette référence — pas « environ » ;
  3. écrivez la phrase qui dit pourquoi ce n'est pas une bonne nouvelle, et ce que cela impose à
     tout score annoncé ensuite.

À écrire : votre ligne de calcul, le score du modèle constant, et la phrase.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : votre référence, en une ligne, sans réutiliser `reference`.
# ma_reference = ...
#
# TODO : le modèle constant, et son score.
# pred = ...
# print((pred == yval).mean() * 100)
#

# ── contrôle de FORME : il ne dit aucune valeur attendue.
controle("`ma_reference` est un pourcentage, pas une proportion",
         lambda: 50 < ma_reference <= 100,
         "la classe majoritaire fait plus de la moitié de la table")
controle("le modèle constant ne prédit que des 0 et des 1", lambda: set(pred) <= {0, 1})
controle("le score du modèle constant coïncide avec VOTRE référence",
         lambda: abs((pred == yval).mean() * 100 - ma_reference) < 0.05,
         "les deux mesures doivent tomber ensemble : c est le point de la question")


### Question 2 — regrouper la cible, et l'afficher en barres  ·  *carnet J1*  ·  *cœur*


Trouvez, en regroupant, la colonne qui sépare le mieux les deux réponses.

À faire :
  1. groupez `patient_absent` par AU MOINS DEUX colonnes de mots différentes (`quartier`,
     `sexe`, `assurance`…) — en gardant le nombre de lignes (`size`) et le taux (`mean`) ;
  2. affichez en barres celle qui sépare le mieux ;
  3. justifiez votre choix par le chiffre — et vérifiez que le plus petit groupe reste assez
     gros pour qu'on en conclue quelque chose.

**Le critère, sans lequel il n'y a pas de réponse** : l'écart entre le taux le plus haut et le
taux le plus bas, sur des groupes d'au moins 500 lignes. Sans critère, une colonne à 9 000
valeurs sépare toujours « mieux » — chaque ligne a son propre groupe. Ce n'est pas une
trouvaille, c'est un piège, et il est là exprès.

À écrire : les tableaux, le graphique, le critère appliqué, et la phrase de justification.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : au moins deux regroupements, ici.
# par_colonne = train.groupby("...", observed=True)[CIBLE].agg(lignes="size", taux="mean")
#
# TODO : le graphique en barres, ici.

controle("le regroupement a au moins deux groupes", lambda: par_colonne.shape[0] >= 2)
controle("chaque groupe compte au moins 500 lignes", lambda: par_colonne["lignes"].min() >= 500,
         "le critère du cours : sous 500 lignes, on ne conclut rien")
controle("la colonne des taux est un TAUX, pas un effectif", lambda: par_colonne["taux"].max() <= 1)


### Question 3 — cinq constats chiffrés  ·  *carnet J1*  ·  *cœur*


Cinq constats — dont au moins DEUX que vous mesurez vous-même.

À faire : ajoutez cinq constats avec `constat(...)`. Chacun prend quatre morceaux :
  (1) ce que vous avez remarqué, (2) le chiffre, (3) la comparaison, (4) ce que vous en faites.

Au moins deux de ces constats doivent venir d'une mesure que VOUS avez produite (un
regroupement, un comptage, une moyenne) — pas d'un tableau déjà affiché.

À écrire : les cinq lignes.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : cinq constats, ici.
# constat("ce que j'ai remarqué", "le chiffre", "la comparaison", "ce que j'en fais")

controle("cinq constats sont écrits", lambda: len(CONSTATS) == 5)
controle("chacun a ses quatre morceaux remplis",
         lambda: all(c["chiffre"] and c["comparaison"] and c["conséquence"] for c in CONSTATS))


### Question 4 — trois défauts certains — annoncés, puis mesurés  ·  *carnet J1*  ·  *cœur*


Trois défauts ont été **établis** en séance 1 : un nombre rangé en texte, des lignes
dupliquées, et un âge impossible. Ce sont des **diagnostics certains** — et cela ne dit rien
de ce qu'ils coûtent.

**Announcez AVANT de regarder le bloc « les trois défauts, mesurés » juste en dessous.**
C'est l'ordre du cours : d'abord la phrase, ensuite le chiffre.

À faire, pour chacun des trois :
  1. qu'aurait appris de faux un modèle qui les garderait ? Ce défaut atteint-il le modèle
     d'aujourd'hui ? (`A_ECARTER` et la section 3 de J1 répondent pour l'un d'eux.)
  2. pour l'âge : mesurez l'exactitude de validation avec les âges négatifs mis à `NaN`, puis
     sans cette correction, et comparez l'écart à `erreur_type` ;
  3. concluez en deux lignes : un défaut **certain** qui ne bouge pas le score est-il pour
     autant sans importance ?

À écrire : trois annonces, puis les deux scores, leur écart face à l'erreur-type, la conclusion.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : vos trois annonces, ICI, avant de descendre.
#
# TODO (après avoir lu le bloc ci-dessous) : la mesure sur l'âge, ici.
# import numpy as np
# def score_age(corriger):
#     """Exactitude de validation, avec ou sans la correction des âges impossibles."""
#     Xa = X.copy()
#     if corriger:
#         for col in ("age", "age_mois"):
#             Xa.loc[Xa[col] < 0, col] = np.nan
#     a, b, ya, yb = train_test_split(Xa, y, test_size=0.2, stratify=y, random_state=0)
#     return 100 * HistGradientBoostingClassifier(random_state=0).fit(a, ya).score(b, yb)
# s1, s2 = score_age(True), score_age(False)
# print(f"{s1:.2f} %  contre  {s2:.2f} %   (erreur-type ± {erreur_type:.2f})")

controle("les trois défauts sont annoncés AVANT leurs mesures", lambda: len(PREDICTIONS) >= 3,
         "la question demande trois phrases avant de regarder les correctifs")


### Les trois défauts, mesurés devant vous

Vous avez annoncé : maintenant on regarde. Ces trois cellules étaient en haut du carnet —
elles sont **descendues ici** pour que la réponse ne soit pas sous vos yeux au moment où
vous écrivez.


In [ ]:
taille = pd.to_numeric(train["taille_cm"].str.replace(",", "."), errors="coerce")

bilan = pd.DataFrame({
    "valeurs": [int(taille.notna().sum()), int(taille.isna().sum())],
}, index=["converties", "non convertibles"])
print("la colonne de taille, après conversion \n")
display(bilan)
print("type obtenu :", taille.dtype)

# LA CONSÉQUENCE : avant, on ne pouvait que compter des étiquettes ; après, on peut COMPARER
avant_apres = pd.DataFrame({
    "valeurs distinctes": [train["taille_cm"].nunique(), taille.nunique()],
    "plus petite": [min(train["taille_cm"]), f"{taille.min():.1f} cm"],
    "plus grande": [max(train["taille_cm"]), f"{taille.max():.1f} cm"],
}, index=["rangée en texte", "convertie en nombre"])
print("ce que la conversion change \n")
display(avant_apres)
print(f"→ un tri sur du texte range « 99 » après « 100 » : "
      f"{min(train['taille_cm'])} est la plus PETITE chaîne, pas la plus petite taille.")


In [ ]:
avant = len(train)
train_sans_doublons = train.drop_duplicates()
bilan = pd.DataFrame({"lignes": [avant, len(train_sans_doublons), avant - len(train_sans_doublons)]},
                     index=["avant", "après", "retirées"])
print("les doublons retirés \n")
display(bilan)
print("→ rien à retirer. La correction est un contrôle, et ce contrôle a rendu zéro.")


In [ ]:
# les deux colonnes, et ce qu'elles laissent voir
impossible = train["age"] == -1
age_mois_impossible = train["age_mois"] < 0
sans_naissance = train["date_naissance"].isna()

bilan = pd.DataFrame({
    "lignes concernées": [int(impossible.sum()), int(age_mois_impossible.sum())],
    "date de naissance absente sur ces mêmes lignes":
        [int(train.loc[impossible, "date_naissance"].isna().sum()),
         int(train.loc[age_mois_impossible, "date_naissance"].isna().sum())],
}, index=["age", "age_mois"]).T
print("les âges impossibles \n")
display(bilan.rename(columns={0: "valeur"}))

meme_lignes = set(train.index[impossible]) == set(train.index[sans_naissance])
print("→ la casse laisse une trace ailleurs :",
      "OUI — la date de naissance manque sur les MÊMES lignes" if meme_lignes
      else "NON — la trace n'est pas là, et il faut le dire ainsi")


### Question 5 — deux réglages, et la barre qui décide  ·  *carnet J1*  ·  *cœur*


Deux réglages, un seul tableau. Ce que vous cherchez n'est pas « quel réglage gagne », mais
« qu'est-ce que la barre de bruit me laisse dire de chacun ? »

À faire, et à mettre dans UN SEUL tableau — une ligne par essai :
  1. la taille du jeu de validation : refaites le découpage ET l'entraînement avec AU MOINS DEUX
     tailles (20 % et 50 %, par exemple). Pour CHAQUE taille, recalculez la barre : racine de
     p(1-p)/n, avec le n de CE découpage — la barre change avec lui ;
  2. le pas d'apprentissage : AU MOINS TROIS valeurs de `learning_rate` (0,05 / 0,5 / 2), en ne
     touchant à rien d'autre. Notez le score ET le nombre d'arbres effectivement construits
     (`m.n_iter_`) ;
  3. pour chaque ligne, ajoutez l'écart à la référence, puis cet écart divisé par la barre de
     la ligne : combien de barres de bruit ?
  4. montrez-le : le tableau, puis deux barres côte à côte — le score, avec sa barre d'erreur
     dessus. L'écart et le bruit se lisent mieux sur une figure qu'en deux nombres ;

**Pourquoi un seul tableau** : les deux réglages posent la même question — cet écart est-il
plus grand que le bruit du tirage ? Un tableau qui les range côte à côte rend la comparaison
visible ; deux tableaux séparés la cachent.

À écrire : le tableau, puis deux conclusions — laquelle des deux tailles donne la mesure la
plus fiable (et pourquoi c'est une histoire de `n`), et ce que le pas d'apprentissage vous a
appris sur ce qui borne vraiment ce modèle.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : un seul tableau, une ligne par réglage.
# reglages = []
#   pour chaque taille de validation : le score, la barre recalculee, l'ecart en barres
#   pour chaque learning_rate       : le score, le nombre d'arbres, l'ecart en barres
# display(pd.DataFrame(reglages))
#
# TODO : les deux conclusions.
# essais = []                   (une ligne par réglage essayé)

controle("le tableau d essais a une ligne par essai, au moins cinq", lambda: len(essais) >= 5)
controle("chaque ligne porte un écart rapporté à la barre de CETTE ligne",
         lambda: all("barres" in c or "barre" in c for c in essais[0].keys()),
         "l écart divisé par la barre : ce qui rend deux réglages comparables")


### Question 6 — ce que vous livrez : le fichier et la fonction  ·  *carnet J1*  ·  *approfondissement*


Deux objets quittent votre poste : le fichier de soumission, et la fonction `predict`. Contrôlez
les deux.

À faire :
  1. le fichier — affichez les premières lignes de `submission.csv` et vérifiez sa forme ;
     comptez les `1` prédits et la part qu'ils représentent ; comparez cette part à celle de
     l'historique. Si l'écart dépasse quelques points, dites ce que cela signale — et si vous
     ne le savez pas encore, dites-le : c'est une réponse ;
  2. la fonction — appelez `predict(df)` sur une tranche de la table brute : elle doit répondre ;
  3. cassez-la DEUX fois, de deux façons différentes (par exemple retirer une colonne qu'elle
     utilise, puis en ajouter une qu'elle ne connaît pas) ;
  4. pour chacune, dites ce qui vous a prévenu — le test, ou le modèle.

**Le lien entre les deux** : ce sont les deux seules choses qui vous survivront. Un fichier
mal formé et une fonction qui casse en silence sont les deux façons de rendre un travail juste
et inutilisable.

À écrire : la forme et les deux parts du fichier, les deux cassures, et qui a prévenu à chaque
fois.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : le fichier — sa forme, les 1, la part, la comparaison.
# print(pd.read_csv("submission.csv").head())
# ...
#
# TODO : la fonction — l'appel normal, puis deux cassures.
# predict(test.head(500))
# ...

controle("le fichier a exactement deux colonnes", lambda: list(soumission.columns) == [ID, CIBLE])
controle("il porte autant de lignes que le test", lambda: len(soumission) == len(test))
controle("la part de 1 prédits est un pourcentage",
         lambda: 0 <= 100 * soumission[CIBLE].mean() <= 100)


### Question 7 — vos propres essais  ·  *carnet J1*  ·  *cœur*


Au moins TROIS essais À VOUS — dont un dont vous attendiez qu'il DÉGRADE le score.

**La règle de l'ordre est appliquée, pas demandée** : `noter(...)` refuse d'enregistrer un
essai pour lequel vous n'avez pas écrit de prédiction. Écrivez `prevoir(...)` d'abord.

À faire :
  1. `prevoir("mon essai", "mieux/pire, et pourquoi", "la raison")` ;
  2. mesurez — ou, pour la garantie complète de l'ordre, mesurez DANS `essai("mon essai",
     fonction)` : il date la mesure lui-même ;
  3. `noter("mon essai", "ce que j'ai changé", "un signe et une raison", score)`.

Un essai dont l'attendu était « pire » et qui l'est vraiment vaut mieux qu'un essai qui monte
par hasard. Et **avant de conclure qu'un essai a amélioré quoi que ce soit**, confrontez le
gain à la barre : en dessous, écrivez « dans le bruit » — c'est une réponse complète.

À écrire : au moins trois lignes, les vôtres.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : trois fois : prevoir(...), puis la mesure, puis noter(...).
# prevoir("mon essai", "mieux, car ...", "parce que ...")
#
# noter("mon essai", "ce que j'ai changé", "mieux/pire, et pourquoi", score)

controle("au moins trois essais À VOUS (les amorces ne comptent pas)",
         lambda: sum(1 for e in ESSAIS
                     if e["hypothèse"] not in ("rien faire du tout", "un modèle suffit-il ?")) >= 3)
controle("chaque essai dit le SIGNE qu il attend",
         lambda: all(any(m in e["attendu"].lower() for m in ("mieux", "pire", "pareil")) for e in ESSAIS))


### Question 8 — ce que le linéaire coûte — en barres de bruit  ·  *carnet J2*  ·  *cœur*


Ce que le linéaire coûte, **en barres de bruit** — et ce qu'il rapporte en échange.

À faire :
  1. l'écart entre les deux familles est affiché par le carnet : **divisez-le par `erreur_type`**.
     Combien de barres de bruit vaut-il ? Est-ce une différence, au sens de la séance ?
  2. lisez **trois** coefficients dans le tableau de la section 1 : nommez-les, donnez leur signe,
     et dites ce que le signe ajoute que la taille ne dit pas ;
  3. concluez en deux lignes : ce que le linéaire coûte en préparation (indicatrices, comblement,
     mise à l'échelle) et ce qu'il rapporte (des coefficients qui se lisent, un par un).

À écrire : le rapport en barres, les trois coefficients avec leur signe, et les deux lignes.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : l'écart divisé par la barre de bruit.
# print(... / erreur_type)
#
# TODO : trois coefficients, leur signe, et ce que le signe ajoute.
# rapport = ...                 (l écart divisé par la barre)
# trois_coefficients = [...]     (trois coefficients lus dans le tableau de la section 1)

controle("le rapport écart / barre est un nombre fini",
         lambda: isinstance(rapport, (int, float)) and abs(rapport) < 1e3)
controle("les trois coefficients lus sont des nombres", lambda: len(trois_coefficients) == 3)


### Question 9 — votre feature, appliquée des deux côtés  ·  *carnet J2*  ·  *approfondissement*


Écrivez une feature, ANNONCEZ ce que vous en attendez, puis mesurez.

À faire :
  1. AVANT de mesurer : écrivez votre feature dans `ajouter_features` — **c'est la seule
     cellule de la mise en place faite pour être modifiée** — et notez ce que vous en attendez
     (un signe ET une raison) ;
  2. `prevoir(...)`, puis `remesurer()` : la chaîne passe par `preparer()`, donc `X` reçoit
     votre feature **une seule fois**, et `modele` est mis à jour ;
  3. appelez `predict(test.head(500))` : elle doit répondre ;
  4. **contournez `preparer()` une fois** : appelez `modele.predict` sur une table que vous
     avez préparée vous-même, à la main, en oubliant une des étapes. Regardez ce qui se passe,
     et notez-le : c'est le bug que la séance 2 décrit.

À écrire : la feature, l'attendu, le mesuré, ce que le contournement a produit, la conclusion.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : votre feature, dans `ajouter_features` (mise en place) :
# df["ma_feature"] = ...
#
# TODO : prevoir(...), remesurer(), puis predict(test.head(500))
#
# TODO : le contournement de preparer(), et ce qu'il a produit
# n_colonnes_avant = X.shape[1]  (AVANT votre feature)
# score_avec, score_contourne = ..., ...

controle("votre feature a enrichi la table", lambda: X.shape[1] > n_colonnes_avant)
controle("l attendu a été écrit AVANT la mesure",
         lambda: len([p for p in PREDICTIONS if p['essai'] == 'ma feature']) == 1)
controle("le contournement a produit autre chose que la chaîne",
         lambda: abs(score_avec - score_contourne) > 1e-9,
         "si les deux se valent, le contournement n a pas contourné grand-chose")


### Question 10 — la colonne qui disparaît : la trouver, et dire ce qu'elle vaudra au classement  ·  *carnet J2*  ·  *cœur*


**Cherchez-la vous-même.** Aucune cellule du carnet ne vous la nomme : l'écart de remplissage
entre l'historique et le test la désigne.

À faire :
  1. comparez, colonne par colonne, le taux de valeurs manquantes de `train` et celui de `test`,
     triez par écart, et dites laquelle se détache — et **de combien de points** ;
  2. **expliquez, à partir de cet écart seul**, ce que la colonne vaudra au classement : quelle
     part du test porte encore une valeur ? Que devient un gain interne de quelques points quand
     il ne reste presque rien à lire ?
  3. dites ce que cela change dans la liste des colonnes écartées de la séance 1.

À écrire : la colonne, l'écart en points, la part du test encore renseignée, et votre conclusion.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : l'écart de remplissage entre train et test, colonne par colonne.
# ...
#
# TODO : la part du test qui porte encore une valeur, et votre conclusion.
# ecarts = ...                   (le tri des écarts de remplissage, train contre test)
# colonne_trouvee, ecart_remplissage = ..., ...

controle("l écart est un pourcentage", lambda: 0 <= ecart_remplissage <= 100)
controle("la colonne nommée sort bien du tri", lambda: colonne_trouvee in list(ecarts.index[:3]),
         "elle doit figurer parmi les plus grands écarts — sinon, ce n est pas celle-là")


### Question 11 — les deux signes, et la règle qui les couvre  ·  *carnet J2*  ·  *cœur*


Le carnet a montré les deux méthodes sur une colonne chacune. À vous de dire ce qu'elles disent
**ensemble**, et jusqu'où la règle va.

À faire :
  1. rédigez en **une phrase** la règle qui attrape les deux familles (le vide qui suffit, et la
     valeur constante qui trahit) ;
  2. **éprouvez cette règle** sur `age`, qui porte lui aussi une valeur constante (−1) sur une
     partie des lignes : est-ce une fuite, ou une mesure manquante ? Justifiez par ce que la
     valeur **veut dire**, pas par le fait qu'elle est constante ;
  3. dites laquelle des deux colonnes trouvées par le carnet vous retireriez de la table, et
     pourquoi l'autre ne pose même pas la question.

À écrire : la règle en une phrase, le verdict sur `age` avec sa justification, et votre choix.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : la règle qui couvre les deux signes, en une phrase.
#
# TODO : `age` et sa valeur constante — fuite, ou mesure manquante ? pourquoi ?
# train.groupby(CIBLE)["age"].describe()
# regle = ...                    (la règle des deux signes, en une phrase)
# verdict_age = ...              (fuite, ou mesure manquante ? et pourquoi)

controle("la règle est une phrase, pas un mot", lambda: isinstance(regle, str) and len(regle) >= 40)
controle("la règle cite les DEUX signes",
         lambda: ("vide" in regle.lower() or "trou" in regle.lower()) and "valeur" in regle.lower())
controle("le verdict sur `age` est argumenté", lambda: isinstance(verdict_age, str) and len(verdict_age) >= 30)


### Question 12 — le facteur de confusion : le rappel, à délai égal  ·  *carnet J2*  ·  *approfondissement*


Le plus transportable de la séance. Un rappel est envoyé, et l'absence est **plus fréquente
avec** le rappel. Faut-il en conclure que le rappel fait manquer les rendez-vous ?

À faire :
  1. calculez le taux d'absence avec rappel et sans rappel, globalement — c'est le chiffre
     qui accuse ;
  2. refaites-le **à délai égal** : par tranches de délai (`pd.cut` sur `delai_jours`, comme en
     séance 1), le taux d'absence avec et sans rappel dans CHAQUE tranche ;
  3. comparez : l'écart garde-t-il son signe ? Regardez aussi les effectifs — toutes les
     tranches ne portent pas les deux groupes ;
  4. nommez la variable qui explique les deux, et dites laquelle des deux mesures est la
     bonne.

À écrire : l'écart global, les écarts par tranche, et la conclusion en trois lignes.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : l'écart global, ici.
# taux = train.assign(rappel=train["rappel_envoye"].fillna("non envoyé"))
# display((100 * taux.groupby("rappel", observed=True)[CIBLE].mean()).round(2))
#
# TODO : puis les mêmes taux PAR TRANCHE de délai.
# taux = ...                     (les deux taux d absence, avec et sans rappel)
# effectifs = ...                (le nombre de lignes de chaque groupe comparé)

controle("les taux sont des pourcentages", lambda: taux.max() <= 100 and taux.min() >= 0)
controle("aucun groupe comparé ne fait moins de 500 lignes", lambda: effectifs.min() >= 500,
         "la règle du cours vaut aussi pour un facteur de confusion")


### Question 13 — le rapport final — les trois tableaux  ·  *carnet J1 + J2*  ·  *cœur*


Les trois tableaux se remplissent **dans les questions** : les `CONSTATS` par la question 3,
les `ESSAIS` par la question 7, et les `HYPOTHÈSES` — personne ne les a encore écrites :
c'est cette question qui les produit.

À faire :
  1. écrivez **trois** hypothèses de features avec `hypothese(...)`. L'une d'elles peut être la
     feature de la question 9. Chacune dit : ce qu'elle raconte (avec le **signe** attendu),
     comment on la teste, et ce qui la **contredirait** — un résultat précis, pas « si ça ne
     marche pas » ;
  2. vérifiez que `CONSTATS` en compte bien cinq — dont au moins deux mesurés par vous ;
  3. vérifiez que `ESSAIS` en compte bien trois **à vous**, chacun avec un signe ET une raison.

Relancer une cellule **remplace** sa ligne : vous pouvez recommencer sans rien doubler. Si
vous relancez tout le carnet, la mise en place vide les trois tableaux avant de les remplir.

À écrire : les trois hypothèses, puis les trois tableaux tels qu'ils sont.


In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : trois hypothèses, ici.
# hypothese("ma feature", "ce qu'elle raconte, avec le signe", "comment je la teste",
#           "ce qui la contredirait")

controle("les trois tableaux sont remplis avant le rapport",
         lambda: len(CONSTATS) == 5 and len(HYPOTHESES) == 3 and len(ESSAIS) >= 3)


## Les trois tableaux à rendre

C'est la pièce principale du devoir. Les fonctions `constat(...)`, `hypothese(...)` et
`noter(...)` sont déjà définies dans la mise en place : appelez-les, elles affichent le tableau.

### 1. Cinq constats chiffrés

Chaque ligne a **quatre morceaux** : l'observation, le chiffre, **par rapport à quoi**, et ce que
vous en faites. Un constat sans comparaison ne compte pas.

### 2. Trois hypothèses de features

Une hypothèse est une **prédiction qu'on peut tester** : ce qu'elle raconte, comment on la
teste, et **ce qui la contredirait**. Une hypothèse qu'aucun résultat ne pourrait contredire ne
vaut rien.

### 3. Le tableau d'essais

Une ligne par changement essayé. L'**attendu** porte un signe **et** une raison.

La cellule qui suit affiche les trois tableaux tels que vous les rendez.


**Ce qu'on rend, et pourquoi ce format.** Ce qu'on rend n'est pas un carnet : c'est un **texte**.
« Le modèle fait 82 % » ne dit rien ; « la cible est déséquilibrée à 18 %, donc 82 % vaut à peine
mieux que la référence » déclenche une décision. Entre les deux, la différence n'est pas le
chiffre : c'est la comparaison — et c'est exactement ce que les trois tableaux ci-dessus
demandent.


In [ ]:
print("CONSTATS CHIFFRÉS \n")
display(pd.DataFrame(CONSTATS))
print("HYPOTHÈSES DE FEATURES \n")
display(pd.DataFrame(HYPOTHESES))
print("TABLEAU D'ESSAIS \n")
display(pd.DataFrame(ESSAIS))

# ── la relecture : des contrôles de FORME, qui ne révèlent aucune valeur.
#    Les deux essais du haut du carnet sont des AMORCES : ils ne comptent pas comme les vôtres.
AMORCES = ('rien faire du tout', 'un modèle suffit-il ?')
print("LA RELECTURE \n")
verifier('cinq constats', len(CONSTATS) == 5,
         'la question 3 en demande cinq — comptez-les')
verifier('chaque constat porte un chiffre ET une comparaison',
         all(c['chiffre'] and c['comparaison'] for c in CONSTATS),
         'un constat sans comparaison ne compte pas : « par rapport à quoi ? »')
verifier('au moins deux constats mesurés par vous',
         sum(1 for c in CONSTATS if c['observation'] not in (
             'une colonne ne dit rien', 'deux colonnes disent la même chose',
             'la cible est déséquilibrée')) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres')
verifier('trois hypothèses', len(HYPOTHESES) == 3,
         'la question 13 en demande trois')
verifier('chaque hypothèse dit ce qui la contredirait',
         all(h['contredite si'].strip() for h in HYPOTHESES),
         'une hypothèse que rien ne pourrait contredire ne vaut rien')
verifier('au moins trois essais à vous — les amorces ne comptent pas',
         sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         'la question 7 en demande trois, de vous')
verifier('chaque essai porte un attendu',
         all(e['attendu'].strip() for e in ESSAIS),
         'un attendu = un signe ET une raison')
verifier('chaque essai a été annoncé AVANT que sa ligne soit écrite',
         all(e['prédiction écrite avant ?'] != '— aucune' for e in ESSAIS),
         'noter() refuse sans prevoir(...) : cet échec ne devrait pas arriver')


## Rendre le travail

1. **Exécutez tout** le carnet (`Run All`), pour que les tableaux soient remplis.
2. **Déposez** ce carnet dans votre dépôt GitHub personnel, **avec** votre déclaration d'usage de
   l'IA, dans le **même commit** :
   - téléchargez le modèle **`DECLARATION_modele.md`** depuis le **dépôt du cours** ;
   - recopiez-le dans votre dépôt sous le nom **`declaration_DM1.md`** ;
   - remplissez-le : les deux scores, ce que vous avez produit seul, ce que vous avez fait
     avec un assistant, et ce que vous ne sauriez pas expliquer.
3. Vérifiez que le lien de votre dépôt est bien à jour dans le formulaire de la classe.

> Un travail non déposé n'existe pas.
